# Evaluation: full model vs. partial model templates  (Best-of)

This notebook loads the *per-item* result CSV files (with the column `condition` = `full` or `partial_best`),
calculates success rates (SR), McNemar tests, median/mean values (with bootstrap CIs), and generates plots (box plots)
for fitness, inlier RMSE, translation and rotation errors, and runtime.


In [ ]:

# Imports
import math
import json
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Matplotlib defaults: avoid specifying colors per instruction
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True
})


In [ ]:

# Parameters
data_paths = {
    "apple": "../results/evaluate_full_vs_partial_templates/apple/results_combined.csv",
    "banana": "../results/evaluate_full_vs_partial_templates/banana/results_combined.csv",
    "bottle": "../results/evaluate_full_vs_partial_templates/bottle/results_combined.csv",
    "orange": "../results/evaluate_full_vs_partial_templates/orange/results_combined.csv",
}
rot_success_deg = 15.0
trans_success_m = 0.02   # 20 mm
out_dir = "../results/evaluate_full_vs_partial_templates/eval_out"
Path(out_dir).mkdir(parents=True, exist_ok=True)
print("Output directory:", out_dir)


In [ ]:

# Utilities

def load_items(paths: Dict[str, str]) -> Dict[str, pd.DataFrame]:
    out = {}
    for item, p in paths.items():
        df = pd.read_csv(p)
        df["item"] = item
        out[item] = df
    return out

def add_success_flag(df: pd.DataFrame, rot_deg_thr: float, trans_m_thr: float) -> pd.DataFrame:
    df = df.copy()
    df["success"] = (df["rotation_error"] <= rot_deg_thr) & (df["translation_error"] <= trans_m_thr)
    return df

def summarize_by_condition(df: pd.DataFrame) -> pd.DataFrame:
    agg = df.groupby(["item","condition"]).agg(
        n=("dataset_id","count"),
        sr=("success","mean"),
        mean_fitness=("fitness","mean"),
        median_fitness=("fitness","median"),
        mean_inlier_rmse=("inlier_rmse","mean"),
        median_inlier_rmse=("inlier_rmse","median"),
        mean_trans_err=("translation_error","mean"),
        median_trans_err=("translation_error","median"),
        mean_rot_err=("rotation_error","mean"),
        median_rot_err=("rotation_error","median"),
        mean_duration_s=("duration","mean"),
        median_duration_s=("duration","median"),
    ).reset_index()
    agg["sr"] = agg["sr"] * 100.0  # percent
    return agg

def mcnemar_exact(b: int, c: int) -> float:
    """Exact two-sided McNemar p-value using binomial distribution.

    b: # FULL success & PARTIAL fail

    c: # FULL fail & PARTIAL success

    """
    n = b + c
    if n == 0:
        return 1.0
    # Two-sided exact p-value
    from math import comb
    k = min(b, c)
    p = 0.0
    for i in range(0, k+1):
        p += comb(n, i) * (0.5 ** n)
    p = 2.0 * p
    return min(1.0, p)

def paired_success_table(df: pd.DataFrame) -> pd.DataFrame:
    # pivot to paired rows per (item, dataset_id)
    wide = (df[["item","dataset_id","condition","success"]]
            .pivot_table(index=["item","dataset_id"], columns="condition", values="success", aggfunc="first"))
    wide = wide.dropna(subset=["full","partial_best"], how="any")
    # compute b, c
    b = int(((wide["full"] == True) & (wide["partial_best"] == False)).sum())
    c = int(((wide["full"] == False) & (wide["partial_best"] == True)).sum())
    pval = mcnemar_exact(b, c)
    return pd.DataFrame({"b_full_only":[b], "c_partial_only":[c], "p_mcnemar":[pval]})

def bootstrap_ci(data: np.ndarray, func=np.median, iters: int = 5000, alpha: float = 0.05) -> Tuple[float,float,float]:
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    if data.size == 0:
        return np.nan, np.nan, np.nan
    stat = func(data)
    n = data.size
    stats = []
    rng = np.random.default_rng(42)
    for _ in range(iters):
        sample = rng.choice(data, size=n, replace=True)
        stats.append(func(sample))
    lo = np.percentile(stats, 100*alpha/2)
    hi = np.percentile(stats, 100*(1 - alpha/2))
    return stat, lo, hi

def boxplot_metric(df: pd.DataFrame, metric: str, title: str, outpath: str):
    # Single figure per plot as required
    plt.figure()
    data = [df[df["condition"]=="full"][metric].dropna().values,
            df[df["condition"]=="partial_best"][metric].dropna().values]
    plt.boxplot(data, labels=["full","partial_best"], showmeans=True)
    plt.title(title)
    plt.xlabel("Bedingung")
    plt.ylabel(metric)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.show()



In [ ]:

# Load and prepare data
dfs = load_items(data_paths)
all_df = pd.concat(dfs.values(), ignore_index=True)
all_df = add_success_flag(all_df, rot_success_deg, trans_success_m)
print("Rows loaded:", len(all_df))
print("Items:", sorted(all_df['item'].unique()))
print("Conditions:", sorted(all_df['condition'].unique()))


In [ ]:

# Per-item summaries
per_item = summarize_by_condition(all_df)
per_item.to_csv(Path(out_dir)/"per_item_summary.csv", index=False)
per_item


In [ ]:

# Global summary
global_summary = (all_df.groupby("condition")
                  .apply(lambda g: pd.Series({
                      "n": len(g),
                      "sr": 100.0*g["success"].mean(),
                      "mean_fitness": g["fitness"].mean(),
                      "median_fitness": g["fitness"].median(),
                      "mean_inlier_rmse": g["inlier_rmse"].mean(),
                      "median_inlier_rmse": g["inlier_rmse"].median(),
                      "mean_trans_err": g["translation_error"].mean(),
                      "median_trans_err": g["translation_error"].median(),
                      "mean_rot_err": g["rotation_error"].mean(),
                      "median_rot_err": g["rotation_error"].median(),
                      "mean_duration_s": g["duration"].mean(),
                      "median_duration_s": g["duration"].median(),
                  }))
                 ).reset_index()
global_summary.to_csv(Path(out_dir)/"global_summary.csv", index=False)
global_summary


In [ ]:

# McNemar test (paired success) per item and global
rows = []
for it in sorted(all_df["item"].unique()):
    sub = all_df[all_df["item"]==it]
    tab = paired_success_table(sub)
    tab.insert(0, "item", it)
    rows.append(tab)
item_mcnemar = pd.concat(rows, ignore_index=True)

global_tab = paired_success_table(all_df)
global_tab.insert(0, "item", "_ALL_")

mcnemar_all = pd.concat([item_mcnemar, global_tab], ignore_index=True)
mcnemar_all.to_csv(Path(out_dir)/"mcnemar_results.csv", index=False)
mcnemar_all


In [ ]:

# Boxplots for key metrics (global)
boxplot_metric(all_df, "fitness", "Fitness (global)", str(Path(out_dir)/"box_fitness_global.png"))
boxplot_metric(all_df, "inlier_rmse", "Inlier-RMSE (global)", str(Path(out_dir)/"box_rmse_global.png"))
boxplot_metric(all_df, "translation_error", "Translationsfehler (global, m)", str(Path(out_dir)/"box_trans_global.png"))
boxplot_metric(all_df, "rotation_error", "Rotationsfehler (global, deg)", str(Path(out_dir)/"box_rot_global.png"))
boxplot_metric(all_df, "duration", "Laufzeit pro Registrierung (global, s)", str(Path(out_dir)/"box_time_global.png"))


In [ ]:

# Per-item plots (optional)
for it in sorted(all_df["item"].unique()):
    sub = all_df[all_df["item"]==it]
    boxplot_metric(sub, "fitness", f"Fitness ({it})", str(Path(out_dir)/f"box_fitness_{it}.png"))
    boxplot_metric(sub, "inlier_rmse", f"Inlier-RMSE ({it})", str(Path(out_dir)/f"box_rmse_{it}.png"))
    boxplot_metric(sub, "translation_error", f"Translationsfehler ({it}, m)", str(Path(out_dir)/f"box_trans_{it}.png"))
    boxplot_metric(sub, "rotation_error", f"Rotationsfehler ({it}, deg)", str(Path(out_dir)/f"box_rot_{it}.png"))
    boxplot_metric(sub, "duration", f"Laufzeit pro Registrierung ({it}, s)", str(Path(out_dir)/f"box_time_{it}.png"))
print("Saved per-item figures to", out_dir)


In [ ]:

# LaTeX tables for SR and key metrics

def to_latex_table(df: pd.DataFrame, fname: str, caption: str, label: str):
    tex = df.to_latex(index=False, float_format=lambda x: f"{x:.4f}", escape=True, longtable=False)
    path = Path(out_dir)/fname
    path.write_text(tex, encoding="utf-8")
    print("Saved:", path)

# SR per item & condition
sr_tab = per_item[["item","condition","n","sr"]].copy()
to_latex_table(sr_tab, "table_sr.tex", "Erfolgsraten pro Objekt und Bedingung.", "tab:sr")

# Global summary latex
to_latex_table(global_summary, "table_global_summary.tex", "Globale Kennzahlen über alle Objekte.", "tab:global")

# McNemar latex
to_latex_table(mcnemar_all, "table_mcnemar.tex", "McNemar-Ergebnisse (paired Erfolge).", "tab:mcnemar")
